In [10]:
import json
import operator
from pathlib import Path
from typing import TypedDict, Annotated, List, Dict, Any

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

# --- CARICAMENTO DATI DAI FILE JSON ALLEGATI ---
DATA_DIR = Path("data")

with open(DATA_DIR / "claims.json", "r", encoding="utf-8") as f:
    CLAIMS_DB = {c["claim_id"]: c for c in json.load(f)} #

with open(DATA_DIR / "customers.json", "r", encoding="utf-8") as f:
    CUSTOMERS_DB = {c["customer_id"]: c for c in json.load(f)} #

with open(DATA_DIR / "policies.json", "r", encoding="utf-8") as f:
    POLICIES_DB = {p["policy_id"]: p for p in json.load(f)} #

with open(DATA_DIR / "requirements.json", "r", encoding="utf-8") as f:
    REQUIREMENTS_DB = {r["loss_type"]: r for r in json.load(f)} #

llm = ChatOllama(model="llama3.2", temperature=0)


# ==============================================================================
# 2. DEFINIZIONE DEI TOOL PER LE 6 TASK
# ==============================================================================
@tool
def verifica_copertura_e_tipo_polizza(policy_id: str, loss_type: str) -> str:
    """Task 3 e Task 4: Verifica se la polizza è attiva, se copre il tipo di evento (Task 3)

    e restituisce il tipo di assicurazione e prodotto (Task 4).
    """
    policy = POLICIES_DB.get(policy_id) #
    if not policy:
        return f"RIFIUTATO — Polizza '{policy_id}' non trovata a sistema." #
    
    if not policy.get("active", False): #
        return f"RIFIUTATO — La polizza '{policy_id}' risulta DISATTIVA." #
        
    coverage = policy.get("coverage", "") #
    if coverage != loss_type: #
        return f"RIFIUTATO — Il tipo evento '{loss_type}' non è coperto dal tipo garanzia '{coverage}'." #
        
    prodotto = policy.get("product", "Sconosciuto") #
    return f"OK — Polizza attiva. Prodotto: '{prodotto}', Copertura: '{coverage}'." #


@tool
def analizza_documenti(claim_documents: List[str], loss_type: str) -> str:
    """Task 1 e Task 6: Ispeziona i documenti forniti dal cliente (Task 6)

    e verifica se tutti i documenti richiesti sono presenti (Task 1).
    """
    req = REQUIREMENTS_DB.get(loss_type) #
    if not req:
        return f"ERRORE — Tipo evento '{loss_type}' non presente nei requisiti." #
        
    req_docs = set(req.get("required_documents", [])) #
    provided_docs = set(claim_documents) #
    
    mancanti = list(req_docs - provided_docs) #
    
    if mancanti:
        return f"INCOMPLETO — Documenti forniti: {list(provided_docs)}. Documenti mancanti: {mancanti}." #
    
    return f"OK — Documentazione completa. Documenti forniti: {list(provided_docs)}." #


@tool
def calcola_budget_e_limiti(estimated_damage: float, policy_id: str, loss_type: str) -> str:
    """Task 2 e Task 5: Verifica franchigia (tetto minimo) e massimale (tetto massimo) (Task 5),

    e controlla il rientro nel budget di approvazione automatica (Task 2).
    """
    policy = POLICIES_DB.get(policy_id) #
    req = REQUIREMENTS_DB.get(loss_type) #
    
    if not policy or not req:
        return "ERRORE — Dati polizza o requisiti mancanti." #
        
    deductible = policy.get("deductible", 0) #
    max_claim = policy.get("max_claim", 0) #
    auto_max = req.get("automatic_review_max", 0) #
    
    # Check 1: Sotto la franchigia (tetto minimo)
    if estimated_damage <= deductible: #
        return f"RIFIUTATO — Danno stimato ({estimated_damage}€) inferiore o pari alla franchigia/tetto minimo ({deductible}€)." #
        
    # Check 2: Supera il massimale (tetto massimo)
    if estimated_damage > max_claim: #
        return f"RIFIUTATO — Danno stimato ({estimated_damage}€) supera il massimale/tetto massimo della polizza ({max_claim}€)." #
        
    # Check 3: Rientro nel budget per approvazione automatica
    if estimated_damage > auto_max: #
        return f"ESCALATION — Danno stimato ({estimated_damage}€) supera la soglia di approvazione automatica ({auto_max}€). Richiesta revisione manuale." #
        
    return f"OK — Danno stimato ({estimated_damage}€) compreso tra franchigia ({deductible}€) e massimale ({max_claim}€), ed entro la soglia automatica ({auto_max}€)." #


@tool
def get_info_sinistro(claim_id: str) -> str:
    """Estrae i dati di sintesi di un sinistro (policy_id, loss_type, estimated_damage, documents)."""
    claim = CLAIMS_DB.get(claim_id)
    if not claim:
        return f"ERRORE — Sinistro '{claim_id}' non trovato."

    return (
        f"Dati Sinistro '{claim_id}':\n"
        f"- Policy ID: {claim.get('policy_id')}\n"
        f"- Tipo Danno: {claim.get('loss_type')}\n"
        f"- Danno Stimato: {claim.get('estimated_damage')} EUR\n"
        f"- Documenti Inviati: {claim.get('documents', [])}"
    )


tools_assicurativi = [
    get_info_sinistro,
    verifica_copertura_e_tipo_polizza,
    analizza_documenti,
    calcola_budget_e_limiti,
]


# ==============================================================================
# 3. PATTERN AGENT: CLASSE REACT RIUTILIZZABILE (LANGGRAPH)
# ==============================================================================

class Agent:
    """Agente ReAct con supporto a StateGraph e MemorySaver."""

    def __init__(self, model, tools, checkpointer=None, system=""):
        self.system = system

        class AgentState(TypedDict):
            messages: Annotated[list[AnyMessage], operator.add]

        self.AgentState = AgentState

        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_llm)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)

        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state):
        return len(state["messages"][-1].tool_calls) > 0

    def call_llm(self, state):
        messages = state["messages"]
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {"messages": [message]}

    def take_action(self, state):
        tool_calls = state["messages"][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"  🔧 Chiamata tool: {t['name']}({t['args']})")
            if t["name"] not in self.tools:
                result = f"Tool '{t['name']}' non esiste. Disponibili: {list(self.tools.keys())}"
            else:
                result = self.tools[t["name"]].invoke(t["args"])
            results.append(ToolMessage(
                tool_call_id=t["id"],
                name=t["name"],
                content=str(result)
            ))
        return {"messages": results}



# ==============================================================================
# 4. ISTANZIAZIONE DELL'AGENTE E SYSTEM PROMPT
# ==============================================================================

system_prompt = """Sei l'Assistente Pratiche Assicurative.
Processa la richiesta di valutazione di un sinistro seguendo rigorosamente questi passi:

1. Chiama `get_info_sinistro` usando il claim_id fornito per ottenere le informazioni di base.
2. Chiama `verifica_copertura_e_tipo_polizza` per accertarti che la polizza sia attiva e che copra il danno.
3. Chiama `analizza_documenti` per verificare se la documentazione inviata è completa.
4. Chiama `calcola_budget_e_limiti` per verificare che l'importo rientri tra franchigia, massimale e soglia automatica.
5. Formula la risposta finale spiegando l'esito chiaro (OK/APPROVATO, ESCALATION, RIFIUTATO) e le motivazioni dettagliate."""


memory = MemorySaver()
agent_app = Agent(
    model=llm,
    tools=tools_assicurativi,
    checkpointer=memory,
    system=system_prompt
)


# ==============================================================================
# 5. ESECUZIONE DELLE RICHIESTE
# ==============================================================================

def esegui_richiesta(testo_richiesta: str, thread_id: str = "default_thread"):
    config = {"configurable": {"thread_id": thread_id}}
    messages = [HumanMessage(content=testo_richiesta)]

    print(f"\n{'═'*60}")
    print(f"🚀 Richiesta: {testo_richiesta}")
    print(f"{'═'*60}")

    for event in agent_app.graph.stream({"messages": messages}, config):
        for node_val in event.values():
            last_msg = node_val["messages"][-1]
            if isinstance(last_msg, HumanMessage):
                continue
            if not getattr(last_msg, "tool_calls", None) and last_msg.content:
                print(f"\n🤖 Risposta finale:\n{last_msg.content}")


# --- ESECUZIONE TEST SUI SINISTRI ---
if __name__ == "__main__":
    # Test 1: Sinistro CL001 (Valido e Approvabile)
    esegui_richiesta(
        "Ciao, puoi valutare la pratica per il sinistro CL001?",
        thread_id="pratica_CL001"
    )

    # Test 2: Sinistro CL002 (Documenti mancanti & Importo elevato)
    esegui_richiesta(
        "Analizza il sinistro CL002 e dimmi se possiamo approvarlo.",
        thread_id="pratica_CL002"
    )

    # Test 3: Sinistro CL003 (Polizza Disattiva)
    esegui_richiesta(
        "Qual è lo stato di approvazione per la pratica CL003?",
        thread_id="pratica_CL003"
    )


════════════════════════════════════════════════════════════
🚀 Richiesta: Ciao, puoi valutare la pratica per il sinistro CL001?
════════════════════════════════════════════════════════════
  🔧 Chiamata tool: get_info_sinistro({'claim_id': 'CL001'})

🤖 Risposta finale:
Dati Sinistro 'CL001':
- Policy ID: POL001
- Tipo Danno: collision
- Danno Stimato: 1200 EUR
- Documenti Inviati: ['photo', 'police_report']
  🔧 Chiamata tool: verifica_copertura_e_tipo_polizza({'claim_id': 'CL001', 'policy_id': 'POL001', 'danno': 'collision'})


ValidationError: 1 validation error for verifica_copertura_e_tipo_polizza
loss_type
  Field required [type=missing, input_value={'claim_id': 'CL001', 'po...', 'danno': 'collision'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

# test dellì'agente

In [ ]:
# --- TEST INTEGRATO AGENTE REACT ---

# Case 1: Sinistro lineare idoneo all'approvazione automatica
esegui_richiesta("Valuta la fattibilità del sinistro CL001", thread_id="t1")

# Case 2: Sinistro con documenti mancanti e importo sopra la soglia
esegui_richiesta("Analizza la pratica del sinistro CL002", thread_id="t2")

# Case 3: Sinistro collegato a una polizza non attiva
esegui_richiesta("Posso procedere con il rimborso per il sinistro CL003?", thread_id="t3")

## Test di verifica trasparenza e correttezza dati estratti

In [4]:
# TEST DI VERIFICA TRASPARENZA E CORRETTEZZA DATI ESTRATTI

def verifica_correttezza_tool(claim_id_da_testare: str):
    print(f"\n============================================================")
    print(f" 🔎 VERIFICA ESTRAZIONE E RISPOSTA PER IL SINISTRO: {claim_id_da_testare}")
    print(f"============================================================")
    
    # 1. Recuperiamo il sinistro dal DB
    claim = CLAIMS_DB.get(claim_id_da_testare)
    if not claim:
        print(f"Errore: Il sinistro {claim_id_da_testare} non esiste nei dati!")
        return

    # Estrazione campi dal sinistro
    policy_id = claim["policy_id"]
    loss_type = claim["loss_type"]
    estimated_damage = claim["estimated_damage"]
    claim_docs = claim["documents"]

    # Recuperiamo i dati corrispondenti dagli altri file JSON
    policy = POLICIES_DB.get(policy_id, {})
    requirement = REQUIREMENTS_DB.get(loss_type, {})

    # STAMPA DEI DATI REALI CONTENUTI NEI JSON 
    print("\n 1. DATI GREZZI PRESENTI NEI FILE JSON:")
    print(f"   • Sinistro ({claim_id_da_testare}): Danno={estimated_damage}€, Documenti Inviati={claim_docs}")
    print(f"   • Polizza ({policy_id}): Attiva={policy.get('active')}, Prodotto={policy.get('product')}, Copertura={policy.get('coverage')}")
    print(f"   • Regole Polizza: Franchigia={policy.get('deductible')}€, Massimale={policy.get('max_claim')}€")
    print(f"   • Requisiti ({loss_type}): Documenti Richiesti={requirement.get('required_documents')}, Max Automatico={requirement.get('automatic_review_max')}€")

    # STAMPA DELLE RISPOSTE GENERATE DAI TOOL
    print("\n 2. RISPOSTE RESTITUITE DAI TUOI TOOL:")

    # Test Tool 1: Copertura e Tipo Polizza
    res_copertura = verifica_copertura_e_tipo_polizza.invoke({
        "policy_id": policy_id, 
        "loss_type": loss_type
    })
    print(f"   ► Tool Copertura:   '{res_copertura}'")

    # Test Tool 2: Documenti
    res_docs = analizza_documenti.invoke({
        "claim_documents": claim_docs, 
        "loss_type": loss_type
    })
    print(f"   ► Tool Documenti:   '{res_docs}'")

    # Test Tool 3: Budget e Limiti
    res_budget = calcola_budget_e_limiti.invoke({
        "estimated_damage": estimated_damage, 
        "policy_id": policy_id, 
        "loss_type": loss_type
    })
    print(f"   ► Tool Budget:      '{res_budget}'")

# --- ESEGUIAMO IL TEST SUI DIVERSI SINISTRI DEL VOSTRO PROGETTO ---
# Test 1: Sinistro CL001 (Caso perfetto)
verifica_correttezza_tool("CL001")

# Test 2: Sinistro CL002 (Caso con documenti mancanti e importo alto)
verifica_correttezza_tool("CL002")

# Test 3: Sinistro CL003 (Caso polizza disattiva)
verifica_correttezza_tool("CL003")


 🔎 VERIFICA ESTRAZIONE E RISPOSTA PER IL SINISTRO: CL001

 1. DATI GREZZI PRESENTI NEI FILE JSON:
   • Sinistro (CL001): Danno=1200€, Documenti Inviati=['photo', 'police_report']
   • Polizza (POL001): Attiva=True, Prodotto=Auto, Copertura=collision
   • Regole Polizza: Franchigia=250€, Massimale=15000€
   • Requisiti (collision): Documenti Richiesti=['photo', 'police_report'], Max Automatico=3000€

 2. RISPOSTE RESTITUITE DAI TUOI TOOL:
   ► Tool Copertura:   'OK — Polizza attiva. Prodotto: 'Auto', Copertura: 'collision'.'
   ► Tool Documenti:   'OK — Documentazione completa. Documenti forniti: ['photo', 'police_report'].'
   ► Tool Budget:      'OK — Danno stimato (1200.0€) compreso tra franchigia (250€) e massimale (15000€), ed entro la soglia automatica (3000€).'

 🔎 VERIFICA ESTRAZIONE E RISPOSTA PER IL SINISTRO: CL002

 1. DATI GREZZI PRESENTI NEI FILE JSON:
   • Sinistro (CL002): Danno=7000€, Documenti Inviati=['photo']
   • Polizza (POL002): Attiva=True, Prodotto=Home, Copertu

## Creazione di un sotto-agente ReAct

In [7]:
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, ToolMessage

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

In [ ]:
import json
import operator
from pathlib import Path
from typing import TypedDict, Annotated

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver


# 1. Setup e Caricamento Dati
llm = ChatOllama(model="llama3.2", temperature=0)
DATA_DIR = Path("data")

with open(DATA_DIR / "claims.json") as f:
    CLAIMS_DB = {c["claim_id"]: c for c in json.load(f)}

with open(DATA_DIR / "policies.json") as f:
    POLICIES_DB = {p["policy_id"]: p for p in json.load(f)}

with open(DATA_DIR / "requirements.json") as f:
    REQUIREMENTS_DB = {r["loss_type"]: r for r in json.load(f)}

with open(DATA_DIR / "customers.json") as f:
    CUSTOMERS_DB = {c["customer_id"]: c for c in json.load(f)}


# 2. Definizione dei Tool per la gestione del sinistro
@tool
def get_dettagli_sinistro(claim_id: str) -> str:
    """Restituisce i dettagli del sinistro, della polizza e del cliente associato.

    Args:
        claim_id: ID del sinistro (es. 'CL001', 'CL002')
    """
    claim = CLAIMS_DB.get(claim_id)
    if not claim:
        return f"Sinistro '{claim_id}' non trovato nel sistema."

    policy = POLICIES_DB.get(claim["policy_id"], {})
    customer = CUSTOMERS_DB.get(policy.get("customer_id"), {})

    return (
        f"--- DETTAGLI SINISTRO {claim_id} ---\n"
        f"Tipo Danno: {claim.get('loss_type')}\n"
        f"Danno Stimato: {claim.get('estimated_damage')} EUR\n"
        f"Documenti allegati: {', '.join(claim.get('documents', []))}\n\n"
        f"--- DETTAGLI POLIZZA {policy.get('policy_id', 'N/A')} ---\n"
        f"Prodotto: {policy.get('product')}\n"
        f"Stato Polizza: {'Attiva' if policy.get('active') else 'Inattiva'}\n"
        f"Copertura: {policy.get('coverage')}\n"
        f"Franchigia: {policy.get('deductible')} EUR\n"
        f"Massimale (Max Claim): {policy.get('max_claim')} EUR\n\n"
        f"--- DETTAGLI CLIENTE ---\n"
        f"Nome: {customer.get('name', 'N/A')}\n"
        f"Livello Rischio: {customer.get('risk_tier', 'N/A')}"
    )


@tool
def verifica_requisiti_e_documenti(claim_id: str) -> str:
    """Verifica se i documenti forniti per il sinistro soddisfano i requisiti previsti per la tipologia di danno.

    Args:
        claim_id: ID del sinistro (es. 'CL001')
    """
    claim = CLAIMS_DB.get(claim_id)
    if not claim:
        return f"Sinistro '{claim_id}' non trovato."

    loss_type = claim.get("loss_type")
    req = REQUIREMENTS_DB.get(loss_type)
    if not req:
        return f"Nessun requisito trovato per il tipo di danno '{loss_type}'."

    doc_presenti = set(claim.get("documents", []))
    doc_richiesti = set(req.get("required_documents", []))
    doc_mancanti = doc_richiesti - doc_presenti

    stato_docs = "Completi" if not doc_mancanti else f"Mancanti ({', '.join(doc_mancanti)})"
    return (
        f"Requisiti Tipo Danno ({loss_type}):\n"
        f"- Documenti richiesti: {', '.join(doc_richiesti)}\n"
        f"- Stato Documenti: {stato_docs}\n"
        f"- Soglia Approvazione Automatica: {req.get('automatic_review_max')} EUR"
    )


@tool
def valuta_pratica_assicurativa(claim_id: str) -> str:
    """Valuta l'esito finale della pratica di sinistro (APPROVATO, ESCALATION, RIFIUTO).

    Args:
        claim_id: ID del sinistro (es. 'CL001')
    """
    claim = CLAIMS_DB.get(claim_id)
    if not claim:
        return f"RIFIUTO — Sinistro '{claim_id}' non trovato."

    policy = POLICIES_DB.get(claim.get("policy_id"))
    if not policy:
        return f"RIFIUTO — Polizza non associata al sinistro {claim_id}."

    if not policy.get("active", False):
        return f"RIFIUTO — La polizza {policy['policy_id']} è inattiva."

    req = REQUIREMENTS_DB.get(claim.get("loss_type"))
    if not req:
        return f"ESCALATION — Impossibile recuperare i requisiti per il danno '{claim.get('loss_type')}'."

    doc_presenti = set(claim.get("documents", []))
    doc_richiesti = set(req.get("required_documents", []))
    doc_mancanti = doc_richiesti - doc_presenti

    if doc_mancanti:
        return (
            f"ESCALATION — Documentazione incompleta per il sinistro {claim_id}. "
            f"Documenti mancanti: {', '.join(doc_mancanti)}."
        )

    stima_danno = claim.get("estimated_damage", 0)
    max_claim = policy.get("max_claim", 0)
    if stima_danno > max_claim:
        return (
            f"RIFIUTO — Il danno stimato ({stima_danno} EUR) supera il massimale previsto "
            f"dalla polizza ({max_claim} EUR)."
        )

    auto_max = req.get("automatic_review_max", 0)
    franchigia = policy.get("deductible", 0)
    importo_liquidabile = stima_danno - franchigia

    if stima_danno > auto_max:
        return (
            f"ESCALATION — Il danno stimato ({stima_danno} EUR) supera la soglia massima per "
            f"la liquidazione automatica ({auto_max} EUR). Richiesta revisione manuale perito."
        )

    return (
        f"APPROVATO — Il sinistro {claim_id} è idoneo all'approvazione automatica. "
        f"Danno stimato: {stima_danno} EUR, Franchigia applicata: {franchigia} EUR. "
        f"Importo netto liquidabile: {importo_liquidabile} EUR."
    )


tools_assicurativi = [get_dettagli_sinistro, verifica_requisiti_e_documenti, valuta_pratica_assicurativa]


# 3. Pattern Agent: Classe ReAct Riutilizzabile
class Agent:
    """Agente ReAct generico con supporto a MemorySaver (checkpointer)."""

    def __init__(self, model, tools, checkpointer=None, system=""):
        self.system = system

        class AgentState(TypedDict):
            messages: Annotated[list[AnyMessage], operator.add]

        self.AgentState = AgentState

        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_llm)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)

        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state):
        """Controlla se il LLM ha richiesto l'esecuzione di tool."""
        return len(state["messages"][-1].tool_calls) > 0

    def call_llm(self, state):
        """Nodo LLM: invoca il modello aggiungendo eventuale system message."""
        messages = state["messages"]
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {"messages": [message]}

    def take_action(self, state):
        """Nodo action: esegue i tool richiesti e restituisce ToolMessage."""
        tool_calls = state["messages"][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"  🔧 Chiamata tool: {t['name']}({t['args']})")
            if t["name"] not in self.tools:
                result = f"Tool '{t['name']}' non esiste. Disponibili: {list(self.tools.keys())}"
            else:
                result = self.tools[t["name"]].invoke(t["args"])
            results.append(ToolMessage(
                tool_call_id=t["id"],
                name=t["name"],
                content=str(result)
            ))
        return {"messages": results}


# 4. Istanziazione dell'Agente con System Prompt Assicurativo
system_prompt = """Sei l'Assistente Pratiche Assicurative.
Il tuo compito è analizzare i sinistri, verificare la presenza dei documenti necessari e determinare l'esito della pratica.

Segui SEMPRE questo ordine per processare ogni richiesta:
1. Chiama `get_dettagli_sinistro` passando l'ID del sinistro (es. 'CL001').
2. Chiama `verifica_requisiti_e_documenti` passando l'ID del sinistro per controllare i documenti allegati e la soglia di stima.
3. Chiama `valuta_pratica_assicurativa` passando l'ID del sinistro per ottenerne la valutazione finale.
4. Formula la risposta finale spiegando chiaramente l'esito (APPROVATO / ESCALATION / RIFIUTO), riportando l'importo liquidabile (se approvato) o i motivi specifici dell'escalation/rifiuto."""

memory = MemorySaver()
agent_app = Agent(
    model=llm,
    tools=tools_assicurativi,
    checkpointer=memory,
    system=system_prompt
)


# 5. Esecuzione
def esegui_richiesta(testo_richiesta: str, thread_id: str = "default_thread"):
    config = {"configurable": {"thread_id": thread_id}}
    messages = [HumanMessage(content=testo_richiesta)]

    print(f"\n{'═'*55}")
    print(f"🚀 Richiesta: {testo_richiesta}")
    print(f"{'═'*55}")

    for event in agent_app.graph.stream({"messages": messages}, config):
        for node_val in event.values():
            last_msg = node_val["messages"][-1]
            if isinstance(last_msg, HumanMessage):
                continue
            if not getattr(last_msg, "tool_calls", None) and last_msg.content:
                print(f"\n🤖 Risposta finale:\n{last_msg.content}")


# Esempi di Test
if __name__ == "__main__":
    # Test Sinistro CL001 (Approvabile)
    esegui_richiesta(
        "Puoi verificare lo stato del sinistro CL001 e dirmi se può essere liquidato?",
        thread_id="pratica_CL001"
    )

    # Test Sinistro CL002 (Escalation per importo elevato / documenti)
    esegui_richiesta(
        "Vorrei aggiornamenti sulla pratica CL002.",
        thread_id="pratica_CL002"
    )


═══════════════════════════════════════════════════════
🚀 Richiesta: Ciao, sono Mario Rossi (E4521). Voglio acquistare il ThinkPad P16s.
═══════════════════════════════════════════════════════
  🔧 Chiamata tool: get_budget_dipendente({'employee_id': 'E4521'})


NameError: name 'EMPLOYEES_DB' is not defined